# wbackdoor — chạy trên Colab (bản chuẩn)

**Trước khi chạy:** Runtime → Change runtime type → **GPU (T4)**.

Chạy các cell **theo thứ tự**. Mỗi cell in ra kiểm tra ở cuối — đọc kỹ trước khi sang cell sau.

Cấu trúc Drive thật (đã xác nhận):
```
MyDrive/wifi_backdoor/
├── wbackdoor/     (code: train_backdoor.py ...)
├── DT-Pose/       (process.py để sinh csi_ap)
└── psba_dataset/  (data_bend.npy — trigger skeleton)
```
Dataset gốc `Person-in-WiFi-3D.zip` nằm ở `MyDrive/wifi-dataset-iclr/` (sửa `ZIP_IN` ở cell 3 nếu khác).

## Cell 1 — Mount Drive + khai báo đường dẫn + kiểm tra

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/wifi_backdoor'
WB   = f'{ROOT}/wbackdoor'
PIW  = '/content/piw3d'   # ổ local Colab — khớp dataset_root trong sweep_colab.yaml
                          # (đọc nhanh hơn Drive; mất khi runtime restart -> chạy lại cell giải nén)
TRIG = f'{ROOT}/psba_dataset/data_bend.npy'
ZIP_IN = '/content/drive/MyDrive/wifi-dataset-iclr/Person-in-WiFi-3D.zip'   # <-- sửa nếu zip ở chỗ khác

print('wbackdoor code :', os.path.exists(f'{WB}/train_backdoor.py'), WB)
print('process.py     :', os.path.exists(f'{PIW}/process.py'), PIW)
print('trigger npy    :', os.path.exists(TRIG), TRIG)
print('dataset zip    :', os.path.exists(ZIP_IN), ZIP_IN)
print('GPU            :', __import__('torch').cuda.get_device_name(0) if __import__('torch').cuda.is_available() else 'CPU (bật GPU đi!)')

## Cell 2 — Cài deps process.py cần
`train_backdoor.py` dùng torch/scipy/numpy/yaml (Colab có sẵn). Chỉ `process.py` cần thêm 2 gói.

In [ ]:
!pip install -q h5py PyWavelets

## Cell 3 — Giải nén dataset gốc vào PIW
Chỉ chạy **1 lần**. Nếu `train_data/` đã có thì bỏ qua cell này. Giải nén 1.6GB mất vài phút.

In [ ]:
import os
os.makedirs(PIW, exist_ok=True)
if os.path.exists(f'{PIW}/train_data') and os.path.exists(f'{PIW}/test_data'):
    print('train_data/test_data đã tồn tại -> BỎ QUA giải nén')
else:
    assert os.path.exists(ZIP_IN), f'Không thấy zip: {ZIP_IN} (sửa ZIP_IN ở cell 1)'
    print('Giải nén', ZIP_IN, '->', PIW, '(chờ vài phút)...')
    !unzip -q -o "$ZIP_IN" -d "$PIW"

print('\n--- cây thư mục sau giải nén ---')
!ls -la "$PIW"
# Nếu thấy 1 folder con thừa (vd Person-in-WiFi-3D/train_data), chạy dòng dưới để kéo lên đúng cấp:
# !mv "$PIW"/Person-in-WiFi-3D/* "$PIW"/ && rmdir "$PIW"/Person-in-WiFi-3D

## Cell 4 — Kiểm tra dataset trước khi process
Xác nhận có đủ `train_data/csi/*.mat` + list.txt. `csi(.mat)` phải ≈ `listed`.

In [ ]:
import os
for split in ['train_data', 'test_data']:
    d = f'{PIW}/{split}'
    lst = f'{d}/{split}_list.txt'
    n_listed = sum(1 for l in open(lst) if l.strip()) if os.path.exists(lst) else 0
    n_mat = len(os.listdir(f'{d}/csi')) if os.path.exists(f'{d}/csi') else 0
    ap = f'{d}/csi_ap'
    n_ap = len(os.listdir(ap)) if os.path.exists(ap) else 0
    print(f'{split}: listed={n_listed}  csi(.mat)={n_mat}  csi_ap(.npy)={n_ap}')
print('\nGhi chú: csi(.mat) << listed  =>  dataset up thiếu / Drive chưa sync xong.')

## Cell 5 — Sinh csi_ap/ từ .mat (process.py)
Bản process.py đã sửa: **skip file .mat thiếu** + **resume** (không làm lại file .npy đã có). Chạy 1 lần; lần sau chạy lại sẽ nhanh vì bỏ qua cái đã xong. Cuối in `done / missing_mat / total`.

In [ ]:
%cd $PIW
!python process.py
%cd $WB

## Cell 6 — Kiểm tra trigger đọc được (data_bend.npy)
File 227MB — nếu vừa up phải đợi Drive sync. Kỳ vọng shape `(1264, 3, 300, 25, 2)`.

In [ ]:
import numpy as np
try:
    a = np.load(TRIG, mmap_mode='r')
    print('OK trigger, shape =', a.shape)
except Exception as e:
    print('LỖI đọc trigger (chưa sync xong?):', e)

## Cell 7 — Train + eval backdoor
Config `attack_colab.yaml` đã trỏ đúng path Drive, `device: null` → tự bắt GPU.
Feeder đã sửa: tự bỏ sample thiếu csi_ap, in `usable samples: X / N`.

In [ ]:
%cd $WB
!python train_backdoor.py --config configs/sweep_colab.yaml

## (Tùy chọn) Cell 8 — Sweep nhiều cấu hình
Chạy quét theta_max × rho để ra bảng kết quả. Bỏ qua nếu chỉ cần 1 run.

In [ ]:
%cd $WB
!python sweep.py --config configs/sweep_colab.yaml